# TabPFN v2: Reproduce & cải tiến — notebook chạy trên Kaggle (2 session)

**Cài đặt Kaggle (thanh bên phải):** Accelerator = **GPU T4 x2**, Internet = **On**
(tài khoản cần xác minh số điện thoại).

| Session | Nội dung | Thời gian dự kiến trên T4 |
|---|---|---|
| **Session 1** | Phần A (reproduce TabPFN v2 vs 6 baseline trên OpenML-CC18) + Phần B-1 (fine-tune + ensemble) | ~4–6 giờ (ước tính) |
| **Session 2** | Phần B-2 (pretrain 6 biến thể nanoTabPFN × 3 seed, đánh giá) + báo cáo cuối | ~3–5 giờ (ước tính) |

Mọi bước đều **chạy tiếp được (resumable)**: nếu session bị dừng, chạy lại đúng các cell đó;
kết quả (dataset, fold, method) và checkpoint đã xong sẽ được bỏ qua.

Cuối mỗi session, file zip chứa toàn bộ kết quả được ghi vào `/kaggle/working/`.
Hãy tải về (hoặc bấm *Save Version → Save & Run All* để Kaggle giữ output).

---
## 0. Cài đặt chung (chạy ở CẢ HAI session)

In [ ]:
# >>>>> EDIT THIS: URL of your GitHub repository <<<<<
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/tabpfn-dl-project.git"

import os
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO = os.path.join(WORK, "tabpfn-dl-project")
os.environ["OPENML_CACHE"] = os.path.join(WORK, "openml_cache")
print(WORK, REPO)

In [ ]:
!nvidia-smi -L
import os
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !cd {REPO} && git pull -q
%cd {REPO}
!pip install -q -r requirements.txt
!python tests/test_smoke.py

In [ ]:
import torch, tabpfn, subprocess, time, sys
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| GPUs", torch.cuda.device_count(), "| tabpfn", tabpfn.__version__)
os.makedirs("logs", exist_ok=True)

def run(cmd, log):
    """Run a shell command, stream its output to the cell and to a log file."""
    print(">>", cmd)
    with open(log, "a") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line); f.write(line)
        p.wait()
    if p.returncode != 0:
        print(f"!! exit code {p.returncode} (see {log})")

def run_parallel(jobs):
    """jobs = [(cmd, gpu_id, logfile), ...] -> one process per GPU, wait for all."""
    procs = []
    for cmd, gpu, log in jobs:
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        print(f">> [GPU {gpu}] {cmd}")
        procs.append((subprocess.Popen(cmd, shell=True, env=env, stdout=open(log, "a"),
                                       stderr=subprocess.STDOUT), log))
    while any(p.poll() is None for p, _ in procs):
        time.sleep(120)
        for _, log in procs:
            lines = open(log).read().strip().splitlines()
            print(f"[{log}] {lines[-1] if lines else ''}")
    print("all jobs finished")

---
# SESSION 1 — Phần A (reproduce) + Phần B-1 (cải tiến ở mức TabPFN v2)

**Phần A** kiểm chứng kết luận chính của Hollmann et al. (Nature 2025): dùng ngay không cần
tune, TabPFN v2 tốt hơn các mô hình cổ điển (cấu hình mặc định) trên dữ liệu bảng nhỏ.
Benchmark = các dataset OpenML-CC18 có ≤ 3000 dòng, ≤ 100 đặc trưng, ≤ 10 lớp;
3-fold stratified CV; metric: ROC-AUC (chính), accuracy, log-loss.

`--val_preds` lưu thêm dự đoán trên tập validation nội bộ, cần cho Phần B-1.

In [ ]:
# ---- Session-1 settings ----
MAX_ROWS     = 3000     # dataset size filter
MAX_DATASETS = 0        # 0 = all datasets that pass the filter
N_FOLDS      = 3
FT_TIME      = 60       # seconds of fine-tuning per fit (B-1). 30 = faster, 120 = stronger
BENCH = "results/benchmark"
common = f"--source openml --max_rows {MAX_ROWS} --n_folds {N_FOLDS} --out {BENCH}"
if MAX_DATASETS: common += f" --max_datasets {MAX_DATASETS}"
print(common)

In [ ]:
# Part A: TabPFN v2 (original Nature checkpoint) vs 6 baselines   (~1 h)
run(f"python scripts/run_benchmark.py {common} --val_preds "
    f"--methods logreg,knn,rf,xgb,lgbm,catboost,tabpfn_v2", "logs/partA.log")

In [ ]:
# Quick look at Part A
!python scripts/make_report.py --bench {BENCH} --nano none --out results/report

### Phần B-1 — làm TabPFN v2 mạnh hơn trên từng dataset

1. **TabPFN v2 + fine-tune**: fine-tune toàn bộ bằng module chính thức `tabpfn.finetuning`,
   cách thích nghi thực tế nhất cho TabPFN v2 theo Rubachev et al. (2025). Giới hạn thời gian
   cố định mỗi lần fit, early stopping trên validation nội bộ.
2. **Ensemble selection hậu kỳ** (Caruana et al. 2004) gồm TabPFN v2, TabPFN v2 đã fine-tune và
   CatBoost. Trọng số chọn trên validation nội bộ (không rò rỉ tập test), rồi áp dụng cho các
   mô hình đã fit lại trên toàn bộ fold train.

In [ ]:
# Part B-1a: fine-tuned TabPFN v2   (~2-4 h with FT_TIME=60)
run(f"python scripts/run_benchmark.py {common} --val_preds "
    f"--methods tabpfn_v2_ft --ft_time_limit {FT_TIME}", "logs/partB1.log")

In [ ]:
# Part B-1b: ensembles from cached predictions   (seconds, CPU)
run(f"python scripts/run_ensembles.py --bench {BENCH}", "logs/partB1_ens.log")

In [ ]:
# Session-1 report + zip
!python scripts/make_report.py --bench {BENCH} --nano none --out results/report
!cd {REPO} && zip -qr {WORK}/session1_results.zip results logs
from IPython.display import Markdown, display
display(Markdown(open("results/report/report.md").read()))
print("-> download", f"{WORK}/session1_results.zip")

---
# SESSION 2 — Phần B-2: foundation model dạng bảng tốt hơn (nanoTabPFN)

Không thể pretrain lại TabPFN v2 thật trên T4 (bản gốc cần 8 GPU × 2 tuần). Vì vậy dùng
**nanoTabPFN**, bản cài đặt thu nhỏ của kiến trúc TabPFN v2, pretrain trên dữ liệu giả lập trong
vài phút, để kiểm tra ba ý tưởng mới có làm foundation model tốt hơn **với cùng chi phí tính toán**:

| Biến thể | Thay đổi |
|---|---|
| `base` | LayerNorm + attention thường + AdamW (baseline nanoTabPFN) |
| `dyt` | Dynamic Tanh thay LayerNorm (CVPR 2025) |
| `gate` | Gated attention: cổng sigmoid sau SDPA (NeurIPS 2025 best paper) |
| `muon` | Optimizer Muon cho các ma trận ẩn (2024/2025) |
| `gate_muon` | gated attention + Muon |
| `full` | DyT + gated attention + Muon |

Mọi biến thể thấy **đúng cùng một chuỗi dataset giả lập** (prior seed cố định), mỗi biến thể 3 seed.

*(Tùy chọn)* Để đưa kết quả Session 1 vào báo cáo cuối: upload `session1_results.zip` thành
Kaggle Dataset, gắn (Add Input) vào notebook này rồi chạy cell kế tiếp.

In [ ]:
# (optional) restore Session-1 results from an attached Kaggle dataset
import glob, zipfile
zips = glob.glob("/kaggle/input/**/session1_results.zip", recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(REPO); print("restored", zips[0])
else:
    found = glob.glob("/kaggle/input/**/results/benchmark/metrics.jsonl", recursive=True)
    if found:
        import shutil
        shutil.copytree(os.path.dirname(found[0]), "results/benchmark", dirs_exist_ok=True)
        print("restored", found[0])
    else:
        print("no Session-1 results attached (Part B-2 still runs)")

In [ ]:
# ---- Session-2 settings ----
STEPS   = 3000          # pre-training steps per run
SEEDS   = "0,1,2"
NANO    = "results/nano"
pre_args = (f"--steps {STEPS} --batch_size 32 --d_model 128 --n_layers 6 --d_ff 256 "
            f"--max_rows 256 --max_features 20 --out {NANO}/ckpt")

# speed check: 60 steps of the baseline -> projected total time
run(f"python scripts/run_nano_pretrain.py --variants base --seeds 99 --steps 60 "
    f"--eval_every 60 --batch_size 32 --d_model 128 --n_layers 6 --d_ff 256 "
    f"--max_rows 256 --max_features 20 --out /tmp/speedcheck", "logs/speed.log")
import torch as _t
h = _t.load("/tmp/speedcheck/base_s99.pt", weights_only=False)
sec = h["train_time_s"] / 60
n_gpu = max(1, _t.cuda.device_count())
print(f"{sec:.3f} s/step -> ~{sec * STEPS * 6 * len(SEEDS.split(',')) / 3600 / n_gpu:.1f} h "
      f"for all runs on {n_gpu} GPU(s) (reduce STEPS if needed)")

In [ ]:
# Part B-2a: pre-train 6 variants x 3 seeds (variants split across GPUs)
import torch as _t
if _t.cuda.device_count() >= 2:
    run_parallel([
        (f"python scripts/run_nano_pretrain.py --variants base,dyt,gate --seeds {SEEDS} {pre_args}",
         0, "logs/pretrain_gpu0.log"),
        (f"python scripts/run_nano_pretrain.py --variants muon,gate_muon,full --seeds {SEEDS} {pre_args}",
         1, "logs/pretrain_gpu1.log"),
    ])
else:
    run(f"python scripts/run_nano_pretrain.py --seeds {SEEDS} {pre_args}", "logs/pretrain.log")

In [ ]:
# Part B-2b: evaluate on the same OpenML-CC18 datasets (+ baselines and real TabPFN v2
# on the identical 200-row context, for reference)
run(f"python scripts/run_nano_eval.py --ckpt_dir {NANO}/ckpt --source openml "
    f"--max_rows 3000 --n_folds 3 --n_ctx 200 --feat_cap 20 --n_ens 4 --with_tabpfn "
    f"--out {NANO}", "logs/nano_eval.log")

In [ ]:
# Final report (Part A + B-1 if restored, B-2) + zip
!python scripts/make_report.py --bench results/benchmark --nano {NANO} --out results/report
!cd {REPO} && zip -qr {WORK}/final_results.zip results logs -x "results/nano/ckpt/*.pt"
from IPython.display import Markdown, display, Image
display(Markdown(open("results/report/report.md").read()))
for f in sorted(glob.glob("results/report/*.png")):
    display(Image(f))
print("-> download", f"{WORK}/final_results.zip")